# 🩺 第十九天 · 医疗大模型评测方法论

**今天目标（约 1.5 小时）**：搞懂「医疗大模型是怎么被打分的」——这是你目标岗位的专业知识，也是明天做《医学问答评测集》的理论基础。

> 昨日回顾：Kaggle 首战提交成功。今天转入核心赛道：**评测（Evaluation）**。
>
> 今天产出：一份 300 字笔记 + 一个用 pandas 写的小评测脚本。

## 1. 为什么医疗大模型需要专门评测

通用模型评测看「聊天爽不爽」，医疗评测看「**敢不敢把命交给它**」。医疗场景三个特殊要求：

| 要求 | 含义 | 例子 |
|---|---|---|
| **准确性** | 医学知识不能错 | 剂量说错 = 事故 |
| **安全性** | 危险建议必须拦截 | 不能教人乱停药 |
| **无幻觉** | 引用必须真实 | 编一篇不存在的指南 = 致命 |

你 D17 做的幻觉侦察，就是评测的雏形——**评测 = 把「挑错」变成一套可重复、可打分的流程**。

## 2. 评测的三个层次

| 层次 | 考什么 | 例子 |
|---|---|---|
L1 基础能力 | 医学知识问答 | 「二甲双胍的一线地位？」 |
L2 专业任务 | 信息抽取、术语归一化、病历理解 | 「从病历里抽出诊断和用药」 |
L3 安全与伦理 | 危险场景拒答、隐私保护 | 「自杀风险如何回应」 |

**注意 L2**：医疗文本任务（NER 实体识别、术语归一化）正是 CBLUE 的核心——你 D12 学的 pandas 筛选/合并，就是处理这类结构化输出。

## 3. 常用评测指标

| 指标 | 怎么算 | 适用场景 |
|---|---|---|
| **准确率 Accuracy** | 答对题数 ÷ 总题数 | 选择题、判断题 |
| 精确率/召回率/F1 | 详见后述 | 信息抽取（抽出的实体对不对、全不全） |
| **一致性 Kappa** | 两位评审对同一批回答打分的一致程度 | 主观题人工评分 |
| **幻觉率** | 回答中虚假引用/事实错误的比例 | 文献引用、数据问答 |
| 安全性 | 危险问题被正确拒答的比例 | 红线题 |

> 你 D15 学的「True=1, False=0」+ `.mean()` 就是算准确率的基础：`(预测 == 答案).mean()`。

## 4. 三大中文医疗评测基准（认识它们）

| 基准 | 全称 | 内容 | 网址 |
|---|---|---|---|
| **CBLUE** | 中文医疗信息处理挑战榜 | 8 类 NLP 任务：实体识别、关系抽取、术语归一化、文本分类、问答… | 阿里天池平台 |
| **MedBench** | 上海人工智能实验室 | 5 大维度、36 个评测集、约 70 万条数据（问答/理解/生成/推理/安全伦理） | medbench.opencompass.org.cn |
| **MedAIBench** | 国家人工智能应用中试基地（医疗） | 基础能力+智慧服务+智慧医疗三层，权威医生出题 | medaibench.cn |

**关键认知**：这些基准的题目 = 医学考试题、临床病历、指南、真实问答——**和你明天要自己造的评测集是同一类素材**。

## 5. 动手：用 pandas 算一个迷你评测（明天作品的地基）

假设你给模型出了 8 道判断题，正确答案和模型回答如下。**先运行下面这个单元格造数据**：

In [1]:
import pandas as pd

eval_df = pd.DataFrame({
    "类别":   ["诊断", "诊断", "用药", "用药", "伦理", "伦理", "指南", "指南"],
    "问题":   ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7", "Q8"],
    "标准答案": ["对", "错", "对", "对", "错", "错", "对", "错"],
    "模型回答": ["对", "对", "对", "错", "错", "对", "对", "错"],
})
eval_df

,类别,问题,标准答案,模型回答
0,诊断,Q1,对,对
1,诊断,Q2,错,对
2,用药,Q3,对,对
3,用药,Q4,对,错
4,伦理,Q5,错,错
5,伦理,Q6,错,对
6,指南,Q7,对,对
7,指南,Q8,错,错


### 练习 1：算「对错列」和总体准确率

```python
eval_df["判对"] = (eval_df["标准答案"] == eval_df["模型回答"])   # True/False
eval_df["判对"].mean()                                          # 准确率
```

In [2]:
# 在这里写你的代
eval_df["判对"] = (eval_df["标准答案"]==eval_df["模型回答"])
eval_df["判对"].mean()

np.float64(0.625)

### 练习 2（检查点）：按「类别」分组算准确率

哪个类别模型表现最差？（提示：`groupby("类别")["判对"].mean()`）

In [4]:
# 在这里写你的代码
eval_df.groupby("类别")["判对"].mean()

类别
伦理    0.5
指南    1.0
用药    0.5
诊断    0.5
Name: 判对, dtype: float64

### 练习 3（加分）：挑出所有判错的题

用布尔筛选找出模型答错的那些行——这就是「错误分析」的第一步，明天评测报告里要写。

In [5]:
# 在这里写你的代码
eval_df[eval_df["标准答案"]!=eval_df["模型回答"]]

,类别,问题,标准答案,模型回答,判对
1,诊断,Q2,错,对,False
3,用药,Q4,对,错,False
5,伦理,Q6,错,对,False


## 6. 阅读任务：写 300 字笔记

打开 **https://medbench.opencompass.org.cn**（走代理），浏览「评测维度」介绍，然后在下面写 300 字回答：

**「医疗大模型是怎么被评分的？有哪些维度？我明天要做的评测集可以借鉴什么？」**

### 我的 300 字笔记

（医疗大模型评测围绕五个维度展开：知识问答、语言理解、语言生成、复杂推理、安全伦理。其中"复杂推理"和"安全伦理"是通用模型评测没有的医疗特色维度。 我明天要做的评测集可以借鉴：① 按维度分类出题（而不是随机出题），这样能算出"每维度得分"；② 借鉴 MedAIBench 的患者/医生双层视角（对患者要通俗、对医生要专业）；③ 设红线题测安全性；④ 借鉴 CBLUE 的客观题设计（有标准答案，便于计算准确率）。

## ✅ D19 完成标准（打钩）

- [ ] 练习 1-3 全部跑通（准确率、分组准确率、错误分析）
- [ ] 300 字笔记写完
- [ ] 保存（**Cmd + S**）

完成后喊我验收。**D20 就是你的核心作品：设计并跑通《医学问答评测集》**——50 道题 + 打分标准 + 用 DeepSeek 评测 + 指标分析。